# Calibration-free MI-BCI on FPGA — Kaggle runner

**Settings (right panel):** Accelerator = **GPU P100** (or T4×2) · **Internet = ON** (needs phone-verified account) · Input: add your dataset containing `hanet_mi.zip`.

1. Interactive test: run cells 1–3 (~10 min).  2. Then **Save Version → Save & Run All (Commit)** — runs everything in the background (browser can be closed).  3. When finished: open the version → **Output** → download `log.txt`, `*.csv`, or attach the output as input to a new version to continue (resume is automatic).

In [ ]:
# 1. Setup: find code, install MOABB, restore previous outputs (resume)
import os, glob, shutil, subprocess, sys, time, zipfile
T0 = time.time()
LIMIT_H = float(os.environ.get('HANET_LIMIT_H', 11.0))   # stop starting new runs after this many hours
IN, BASE = os.environ.get('HANET_IN', '/kaggle/input'), os.environ.get('HANET_BASE', '/kaggle/working')
CODE = f'{BASE}/hanet_mi/python'

# --- code: from the uploaded dataset (extracted folder or zip) ---
if not os.path.exists(CODE):
    src = glob.glob(f'{IN}/**/hanet_mi/python', recursive=True)
    if src:
        shutil.copytree(os.path.dirname(src[0]), f'{BASE}/hanet_mi')
    else:
        z = glob.glob(f'{IN}/**/hanet_mi*.zip', recursive=True)
        assert z, 'Upload hanet_mi.zip as a Kaggle dataset and add it as input'
        zipfile.ZipFile(z[0]).extractall(BASE)
os.chdir(CODE); sys.path.insert(0, CODE)
print('code at', CODE)

# --- resume: copy runs/cache/pretrain/mne_data from attached previous notebook outputs ---
# Kaggle may mount an attached output several levels deep (e.g. /kaggle/input/notebooks/<user>/<name>/),
# so search up to 4 levels and accept a folder only if it sits next to a log.txt or runs/ folder.
print('inputs:', [p for p in glob.glob(f'{IN}/*/*/*') if os.path.isdir(p)][:20])
for d in ('runs', 'cache', 'pretrain'):   # mne_data skipped: thousands of EEG files, very slow to copy
    for depth in range(1, 5):
        for prev in glob.glob(f'{IN}/' + '*/' * depth + d):
            root = os.path.dirname(prev)
            if not (os.path.exists(f'{root}/log.txt') or os.path.isdir(f'{root}/runs')):
                continue
            shutil.copytree(prev, f'{BASE}/{d}', dirs_exist_ok=True); print('restored', prev)
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
print('restored runs:', sorted(os.listdir(f'{BASE}/runs')))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'moabb'], check=False)
os.environ['MNE_DATA'] = f'{BASE}/mne_data'
import mne; mne.set_config('MNE_DATA', f'{BASE}/mne_data', set_env=True)
import torch; print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
# 2. Helpers
PROP = '--no-attention --K1 64 --P1 4 --P2 8 --act elu --dg none'
ATTN = '--K1 64 --P1 4 --P2 8 --act elu --dg none'
EEG, SHAL = '--model eegnet --dg none', '--model shallow --dg none'
CACHE = f'--cache-dir {BASE}/cache'
FT = '--epochs 80 --patience 20'          # fine-tuning from pretrained weights converges faster
LOG = open(f'{BASE}/log.txt', 'a')

def hours(): return (time.time() - T0) / 3600

def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for l in p.stdout:
        if l.startswith(('S0', '[init]', '[data]', '[pretrain]', 'saved', 'loaded', 'train ', 'ep ',
                         'target', 'fold', 'exported', 'Traceback')) or 'error' in l.lower():
            print(l, end='', flush=True); LOG.write(l); LOG.flush()
    p.wait()

def run(name, extra, dataset='2a', subjects=range(1, 10)):
    for s in subjects:
        out = f'{BASE}/runs/{name}/S{s}'
        if os.path.exists(f'{out}/results.json'):
            continue
        if hours() > LIMIT_H:
            print(f'** time limit reached, {name} S{s} left for next session'); return
        sh(f'{sys.executable} -W ignore train_loso.py --dataset {dataset} --targets {s} --out {out} {CACHE} {extra}')
    print(f'== {name} done ({hours():.2f} h)', flush=True)

def pretrain(name, extra, classes='all'):
    out = f'{BASE}/pretrain/{name}.pt'
    if not os.path.exists(out) and hours() < LIMIT_H:
        sh(f'{sys.executable} -W ignore pretrain.py --out {out} --classes {classes} {CACHE} {extra}')
    return out

def table(groups, metric='online_ea', csv=None):
    g = ' '.join(f'--group "{n}={r}"' for n, r in groups)
    c = f'--csv {csv}' if csv else ''
    sh_all = subprocess.run(f'{sys.executable} aggregate.py --base {BASE}/runs --metric {metric} {g} {c}',
                            shell=True, capture_output=True, text=True)
    print(sh_all.stdout + sh_all.stderr); LOG.write(sh_all.stdout); LOG.flush()

def seeds(prefix): return ','.join(f'{prefix}_s{s}' for s in (42, 7, 123))


In [ ]:
# 3. QUICK TEST (interactive session only, ~10 min): checks downloads + one subject end to end.
#    Skip this cell in the background run (it is harmless if left in: results are reused).
QUICK_TEST = False
if QUICK_TEST:
    sh(f'{sys.executable} -W ignore train_loso.py --dataset 2a --targets 1 --epochs 3 --out {BASE}/quicktest {CACHE} {EEG}')
    sh(f'{sys.executable} -W ignore pretrain.py --out {BASE}/quicktest/pt.pt --n-subjects 2 --epochs 1 {CACHE} {PROP}')
    print('QUICK TEST DONE — if you see an S01 line and a "saved" line, start the background run.')


In [ ]:
# 4. QUEUE v3 — final pipeline = Prop CNN + online EA, NO pretraining. Ablations on that model.
#    Results of Version 2 (Prop/EEGNet seeds 7,123 and all 2-class runs) are already in its log.
run('A4_eegnet_s42', f'{EEG} --seed 42')                    # skipped if restored from Version 1
run('A4_prop_s42',   f'{PROP} --seed 42')                   # checkpoints for the int8 export
run('A4_attn_s42',   f'{ATTN} --seed 42')                   # ablation: + hardware attention
run('A4_noreset_s42', f'{PROP} --seed 42 --no-reset-session')  # ablation: no session reset
run('A4_shallow_s42', f'{SHAL} --seed 42')                  # extra baseline
run('A2_shallow_s42', f'{SHAL} --classes lr --seed 42')
for sd in (7, 123):                                          # attention seeds (lowest priority)
    run(f'A4_attn_s{sd}', f'{ATTN} --seed {sd}')
print(f'QUEUE FINISHED in {hours():.2f} h')


In [ ]:
# 5. Tables (also in /kaggle/working/*.csv and log.txt)
G4 = [('Prop', 'A4_prop_s42'), ('EEGNet', 'A4_eegnet_s42'), ('Prop+Attn', seeds('A4_attn')),
      ('Prop noreset', 'A4_noreset_s42'), ('ShallowConvNet', 'A4_shallow_s42'), ('Prop+PT (V1)', 'B4_prop_pt_s42')]
for m in ('online_ea', 'offline_ea_oracle', 'no_ea'):
    table(G4, m, f'{BASE}/table4_{m}.csv')
table([('Shallow', 'A2_shallow_s42')], 'online_ea', f'{BASE}/table2_shallow.csv')


In [ ]:
# 6. int8 quantisation + HLS export of the FINAL model (Prop, no pretraining), all 9 subjects
for s in range(1, 10):
    ck = f'{BASE}/runs/A4_prop_s42/S{s}/hanet_target{s}.pt'
    if os.path.exists(ck) and hours() < LIMIT_H + 0.5:
        sh(f'{sys.executable} -W ignore deploy.py --ckpt {ck} --dataset 2a --target {s} --hls-dir {BASE}/hls_export/S{s}')
shutil.rmtree(f'{BASE}/hanet_mi', ignore_errors=True)   # keep output small (code is in the input dataset)
print('ALL DONE', f'{hours():.2f} h')
